# 02 · Features, models and evaluation (Idea 1)

Thin wrapper around `analysis/` (design: `docs/05_modelling_and_surveillance_plan.md`). Reproduce from the command line:

```bash
cd notebooks/analysis
/data/abar/alexenv/bin/python run_all.py all        # features -> models -> evaluation (~6 min)
/data/abar/alexenv/bin/python run_all.py explain    # SHAP, permutation importance, parsimonious model
/data/abar/alexenv/bin/python run_all.py oracle     # ceiling, definition audit, reference-echo value, mechanism shifts, sample size
/data/abar/alexenv/bin/python run_all.py secondary  # stage 2 -> 3, reintervention breakdown, BVF, gradient slope
```

**What is modelled.** One prediction instance per post-reference echo (the landmark). Outcome: confirmed VARC-3 stage ≥2 SVD, known at the *confirming* echo (C0), with death/endocarditis as competing risk; SVD-driven reinterventions that truncated follow-up before the confirming echo count as events. Primary model: discrete-time cause-specific hazard (boosted multinomial over 6-month periods) → cumulative incidence at any horizon. Comparators: registry-only feature set, no-symptom-flag set, time-since-implant only, implant-time boosted model, cause-specific Cox, the published risk-factor Cox (B4 Table 2 set).

In [ ]:
import json, sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, Markdown, display
OUT = Path('analysis/output'); TAB = OUT / 'tables'; FIG = OUT / 'figures'
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
res = json.load(open(OUT / 'metrics.json'))
display(Markdown('**Usefulness thresholds (docs/03 §0), primary model, confirmed label, 5 y:** ' + json.dumps(res['usefulness'])))

## 1. Discrimination and calibration

Rows: landmark models on *all* test instances and on *currently-negative* instances (the echo itself is not single-echo positive; this is where a surveillance decision is taken), and implant-time models at 5 / 8 / 10 y. AUC and C are cause-specific (competing events are controls) with valve-level bootstrap CIs.

In [ ]:
m = pd.read_csv(TAB / 'metrics.csv')
display(m[['model', 'subset', 'tau', 'n_eligible', 'n_cases', 'AUC', 'AUC_lo', 'AUC_hi', 'C_trunc', 'Brier', 'calib_slope']].round(3))
display(Image(filename=str(FIG / 'auc_models_5y.png'))); display(Image(filename=str(FIG / 'calibration_5y.png')))

## 2. The label is the bottleneck: same instances, same predictions, different definitions

Cases are 'label time ≤ landmark + 5 y' for each definition (prevalent-or-incident), so the risk set is identical across rows.

In [ ]:
display(pd.read_csv(TAB / 'label_sweep_5y.csv').round(3)); display(Image(filename=str(FIG / 'auc_vs_label_5y.png')))
p = TAB / 'oracle_ceiling.csv'
if p.exists():
    display(Markdown('**Achievable ceiling** (same features trained on the latent crossing, evaluated on the oracle label):')); display(pd.read_csv(p).round(3))

## 3. Subgroups, fairness and the held-out new-valve behaviour

Sex has no direct effect in the simulator but acts through labelled size and PPM; the check is calibration and AUC within sex.

In [ ]:
display(pd.read_csv(TAB / 'subgroups_5y.csv').round(3))

## 4. Explainability: SHAP, permutation importance, the parsimonious model, and what one inference returns

In [ ]:
for f in ('shap_summary.png', 'shap_dependence_slope_since_ref.png', 'shap_dependence_ewma_log_mpg.png', 'shap_waterfall_high.png'):
    if (FIG / f).exists(): display(Image(filename=str(FIG / f)))
for t in ('permutation_importance_5y.csv', 'backward_elimination.csv'):
    if (TAB / t).exists(): display(pd.read_csv(TAB / t).round(4).head(20))
if (TAB / 'parsimonious_model.json').exists(): display(Markdown('```json\n' + json.dumps(json.load(open(TAB / 'parsimonious_model.json')), indent=1)[:1500] + '\n```'))
if (TAB / 'example_inference_records.json').exists(): display(Markdown('**Example inference records (CIF, tier, top-3, unconfirmed flag):**')); display(pd.json_normalize(json.load(open(TAB / 'example_inference_records.json'))))

## 5. Cause-specific Cox hazard ratios (per SD) and the oracle-only analyses

In [ ]:
display(pd.read_csv(TAB / 'cox_hr_table.csv').round(3))
for t in ('oracle_definition_audit.csv', 'oracle_reference_value.csv', 'oracle_mechanism_shifts.csv', 'oracle_sample_size.csv', 'secondary_stage2_to_stage3.csv', 'secondary_bvf_reintervention_cif.csv', 'secondary_gradient_slope.csv'):
    if (TAB / t).exists(): display(Markdown(f'**{t}**')); display(pd.read_csv(TAB / t).round(3))
for f in ('oracle_definition_audit.png', 'oracle_sample_size.png'):
    if (FIG / f).exists(): display(Image(filename=str(FIG / f)))